# 03 · Baseline evaluation — Qwen3-ASR-1.7B on four large Vietnamese corpora

Measures the **out-of-the-box** WER/CER of `Qwen/Qwen3-ASR-1.7B-hf` on the
held-out test split of each of four corpora:

| corpus | source | corpus size | streamed here | character |
|---|---|---|---|---|
| `vivoice_full` | [`capleaf/viVoice`](https://huggingface.co/datasets/capleaf/viVoice) | ~1,000 h | 100 h | YouTube speech, native clips |
| `vietspeech` | [`NhutP/VietSpeech`](https://huggingface.co/datasets/NhutP/VietSpeech) | ~1,100 h | 100 h | social-media speech, 2–6 s clips, all three regional accents |
| `vieneu` | [`pnnbao-ump/VieNeu-TTS-140h`](https://huggingface.co/datasets/pnnbao-ump/VieNeu-TTS-140h) | ~140.7 h | 100 h | studio TTS corpus, 193 voices, clean read speech |
| `bud500` | [`linhtran92/viet_bud500`](https://huggingface.co/datasets/linhtran92/viet_bud500) | ~500 h | 100 h | YouTube speech in fixed 2–5 s chunks, no speaker labels |

400 h in total — an equal cap per corpus rather than the whole of any of them,
which is both a budget decision (~46 GB, ~1.5 wall-hours to download) and a
balance decision: the four differ in size by 8x, and taking them in full would
let viVoice and VietSpeech account for 77% of the mixture. Raise
`HOURS_PER_CORPUS` to `None` for the full ~2,740 h; `04_lora_finetune_3ds.ipynb`
must be raised in step.

Equal *hours* is not equal *examples*. Bud500's clips average 2.55 s against
viVoice's 4.14 s, so 100 h of it is 126,326 train clips to viVoice's 71,238 — about
37% of the training examples, and gradient steps are counted per example, not
per hour.

This is the large-corpus counterpart to `01_eval_baseline.ipynb`, which scores a
single 8 h viVoice slice. `04_lora_finetune_3ds.ipynb` fine-tunes on the same
four corpora and diffs against the numbers written here.

## Which VieNeu release, and why

This uses the **140 h** release, not the 1000 h sibling. The 1000 h repo is
gated `manual` and its authors restrict it to institutions, research labs and
universities; this project's request is still awaiting review, so every file
resolves `403`. The 140 h release is gated `auto` — accepting the terms on the
dataset page is enough — and is otherwise the same corpus shape.

Two things worth knowing about it:

- It ships **both** `text` (ordinary Vietnamese) and `phonemized_text` (IPA).
  `corpora.py` pins the text column to `text`: training on phonemes would
  optimise a target the WER metric never sees.
- Audio is 24 kHz and is resampled to 16 kHz on the way in.

It is also the easiest of the four: clean studio read speech, so expect a much
lower baseline WER here than on viVoice or VietSpeech. That makes it the least
informative of the four for judging real-world gains, and the most likely to
show a small delta simply because there is little headroom.

## How the test splits are built

`corpora.py` builds every split here, never at the clip level. Consecutive clips
from one recording share a speaker, a room and a topic, so a random clip-level
split leaks the test set into training. (Bud500 does ship official splits, but
the hour cap takes only a fifth of it, so val and test have to come from the same
slice the training hours did.)
- **viVoice** splits on its real `channel` column.
- **VietSpeech** has no speaker column, so it splits on the recording prefix in
  its filenames (`278_000000086.wav`).
- **VieNeu** splits on its `speaker` column as-is. Counted over the shards it
  holds 193 distinct ids across 74,858 clips — exactly the card's voice count —
  so it is already one id per voice.
- **Bud500** has *no* grouping signal: two columns, `audio` and `transcription`,
  and an empty `path`. It also arrives pre-shuffled at the clip level, so the
  shard index is no proxy either. It falls back to a transcript hash, which keeps
  a repeated sentence out of two splits but lets one speaker sit on both sides.
  **Read its test WER as optimistic** — it is in the mixture for its training
  hours. The other three stay genuinely held out.

See the `corpora` module docstring for the full reasoning.

**Hardware:** RTX 5080 (16 GB) — inference only, ~4 GB VRAM.

Run top-to-bottom (Kernel → Restart & Run All). Writes results to `results/`.

In [1]:
import os, random, numpy as np, torch
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in .env (see .env.example)"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

torch 2.8.0+cu128 | device: cuda | NVIDIA GeForce RTX 5080


## 1 · Build / load the four corpora

Cache-aware: each corpus is streamed once into `data/corpora/<name>/` and reused
after that, so re-running this notebook is free. `HOURS_PER_CORPUS` caps each
corpus at the same number of hours — the corpora differ in size by 8x, and an
equal cap is what stops the two big ones from setting the whole picture.

Keep this cell's `HOURS_PER_CORPUS` and `NAMES` identical to
`04_lora_finetune_3ds.ipynb`. The caches are shared, so whichever notebook runs
first fixes the splits; a mismatch silently scores different clips in the two
runs and voids the before/after diff.

In [2]:
import corpora

# Hours streamed per corpus; None = the whole corpus.
#
# MUST match 04_lora_finetune_3ds.ipynb. The caches are shared, and whichever
# notebook builds first fixes the splits — a mismatch here means the baseline
# and the fine-tune score different clips and the before/after diff is void.
#
# 100 h x 4 = 400 h. Measured costs on this box:
#   disk      115 MB per audio-hour of 16 kHz PCM_16 wav      ->  ~46 GB
#   download  300 audio-hours per wall-hour (shard prefetch)  ->  ~1.5 h to build
HOURS_PER_CORPUS = 100.0
# "vivoice_full" streams viVoice as its native clips and honours the cap above.
# The other entry, "vivoice", is the 8.4 h merged cache notebooks 1-2 use — it
# ignores HOURS_PER_CORPUS entirely, so swap it in only for a quick run whose
# viVoice number is directly comparable to notebook 1 (section 6).
NAMES = ["vivoice_full", "vietspeech", "vieneu", "bud500"]

import shutil
free_gb = shutil.disk_usage(".").free / 1e9
need_gb = 0.115 * len(NAMES) * (HOURS_PER_CORPUS or 750)
print(f"free disk: {free_gb:.0f} GB — cached audio costs ~115 MB per audio-hour, "
      f"so this configuration needs ~{need_gb:.0f} GB\n")

# Probes access first and skips what it cannot reach, so an unapproved gated
# corpus costs a printed line rather than a crash an hour into the build. A
# corpus already built at HOURS_PER_CORPUS is not probed at all — the build
# already proved its bytes are reachable.
built = corpora.prepare_all(NAMES, target_hours=HOURS_PER_CORPUS)
AVAILABLE = list(built)
print("\nusable corpora:", AVAILABLE)

# Stop rather than quietly work on a subset. On 2026-08-04 the access probe hit
# an ImportError for torchcodec — absent from this project's venv — and dropped
# viVoice and VietSpeech, both fully built and stamped on disk. The notebook ran
# on to publish a "baseline" over vieneu alone, in a summary table shaped
# exactly like a complete one. Everything below describes AVAILABLE, and nothing
# below says so.
missing = [n for n in NAMES if n not in AVAILABLE]
if missing:
    raise RuntimeError(
        f"{missing} could not be prepared — see the messages above for why. "
        f"Fix the cause, or set NAMES = {AVAILABLE} to work on that subset on "
        "purpose. 03 and 04 must cover the same corpora: 04 diffs against the "
        "metrics 03 writes.")

free disk: 674 GB — cached audio costs ~115 MB per audio-hour, so this configuration needs ~46 GB

[corpora] vivoice_full: complete cache at data/corpora/vivoice_full — skipping the access probe.


[corpora] vivoice_full: cache hit at data/corpora/vivoice_full (v3, 100 h).
[corpora] vietspeech: complete cache at data/corpora/vietspeech — skipping the access probe.
[corpora] vietspeech: cache hit at data/corpora/vietspeech (v3, 100 h).
[corpora] vieneu: complete cache at data/corpora/vieneu — skipping the access probe.
[corpora] vieneu: cache hit at data/corpora/vieneu (v3, 100 h).
[corpora] bud500: complete cache at data/corpora/bud500 — skipping the access probe.
[corpora] bud500: cache hit at data/corpora/bud500 (v3, 100 h).

usable corpora: ['vivoice_full', 'vietspeech', 'vieneu', 'bud500']


In [3]:
import pandas as pd

rows = []
for name in AVAILABLE:
    d = corpora.load_corpus(name)
    for split in corpora.SPLITS:
        rows.append({"corpus": name, "split": split, "n": len(d[split]),
                     "hours": round(sum(d[split]["duration"]) / 3600, 2),
                     # Speakers, not clips, are what a WER generalizes over: a
                     # test split resting on 2 voices measures those 2 voices.
                     "channels": len(set(d[split]["channel"]))})
comp = pd.DataFrame(rows).pivot(index="corpus", columns="split",
                                values=["n", "hours", "channels"])
print(comp.to_string())

# Only meaningful where a channel signal exists. bud500 has none, so its single
# "" channel is the hash-split fallback doing its job, not a thin test set.
thin = [(r["corpus"], r["split"], r["channels"]) for r in rows
        if r["split"] == "test" and r["channels"] < 5
        and corpora.has_channel_signal(corpora.CORPORA[r["corpus"]])]
for name, split, n in thin:
    print(f"\nWARNING {name}: the {split} split covers only {n} channel(s). Its "
          f"WER describes those speakers, not the corpus — raise HOURS_PER_CORPUS.")
for name in AVAILABLE:
    if not corpora.has_channel_signal(corpora.CORPORA[name]):
        print(f"\nNOTE {name}: no speaker or recording signal exists in this "
              f"corpus, so it is split by transcript hash. One speaker can "
              f"appear in both train and test — read its WER as optimistic.")

for name in AVAILABLE:
    print(f"\n{name}: {corpora.CORPORA[name].note}")
comp

                   n                   hours               channels             
split           test     train     val  test  train    val     test  train   val
corpus                                                                          
bud500        7091.0  126326.0  7001.0  5.06  89.94   5.00      1.0    1.0   1.0
vieneu        4690.0   43451.0  5180.0  8.65  81.61   9.74     19.0  155.0  19.0
vietspeech    6588.0   74014.0  9069.0  7.37  82.53  10.10     99.0  795.0  99.0
vivoice_full  6637.0   71238.0  8990.0  8.99  80.31  10.70     19.0  148.0  19.0

NOTE bud500: no speaker or recording signal exists in this corpus, so it is split by transcript hash. One speaker can appear in both train and test — read its WER as optimistic.

vivoice_full: capleaf/viVoice, ~1,000 h of YouTube speech as native clips (unmerged). Split by its real `channel` column.

vietspeech: NhutP/VietSpeech, ~1,100 h of Vietnamese social media speech (north/central/south accents). Native 16 kHz, lowercase, 

n                   hours               channels         \
split           test     train     val  test  train    val     test  train   
corpus                                                                       
bud500        7091.0  126326.0  7001.0  5.06  89.94   5.00      1.0    1.0   
vieneu        4690.0   43451.0  5180.0  8.65  81.61   9.74     19.0  155.0   
vietspeech    6588.0   74014.0  9069.0  7.37  82.53  10.10     99.0  795.0   
vivoice_full  6637.0   71238.0  8990.0  8.99  80.31  10.70     19.0  148.0   

                    
split          val  
corpus              
bud500         1.0  
vieneu        19.0  
vietspeech    99.0  
vivoice_full  19.0

## 2 · Load the model + processor

In [4]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, attn_implementation="sdpa", device_map=DEVICE,
).eval()
print("loaded", type(model).__name__, model.dtype, model.device)

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

loaded Qwen3ASRForConditionalGeneration torch.bfloat16 cuda:0


## 3 · Evaluate each corpus

`corpora.score_corpus` picks the evaluation rows, transcribes them batched and
scores with the shared `vi_norm` normalizer. Notebook 4 calls the *same*
function with the same `EVAL_LIMIT` and `SEED`, which is what makes the
before/after comparison meaningful — both runs score identical clips.

`EVAL_LIMIT` caps each corpus so a run stays in the tens of minutes. At 100 h
per corpus each test split holds thousands of clips, so all four are
subsampled; the sample is drawn by a seeded shuffle, so it is the same 500 clips
in both notebooks.

In [5]:
import json, pathlib
import pandas as pd

EVAL_LIMIT = 500   # clips per corpus; None = the whole test split
BATCH_SIZE = 8     # lower to 4 if you hit CUDA OOM

processor.tokenizer.padding_side = "left"   # required for correct batched generation

base_metrics, base_frames = {}, {}
for name in AVAILABLE:
    n = len(corpora.eval_rows(name, EVAL_LIMIT))
    print(f"\n=== {name} · {corpora.corpus_dir(name)} [test] · n={n} ===")
    m, f = corpora.score_corpus(model, processor, name, limit=EVAL_LIMIT,
                                batch_size=BATCH_SIZE, seed=SEED)
    base_metrics[name], base_frames[name] = m, f
    print(f"   WER={m['wer']:.4f}  CER={m['cer']:.4f}  "
          f"(legacy WER={m['wer_legacy']:.4f})  n={m['n']}")


=== vivoice_full · data/corpora/vivoice_full [test] · n=500 ===


eval vivoice_full:   0%|          | 0/63 [00:00<?, ?it/s]

   WER=0.0543  CER=0.0270  (legacy WER=0.0713)  n=500

=== vietspeech · data/corpora/vietspeech [test] · n=500 ===


eval vietspeech:   0%|          | 0/63 [00:00<?, ?it/s]

   WER=0.0659  CER=0.0376  (legacy WER=0.0662)  n=500

=== vieneu · data/corpora/vieneu [test] · n=500 ===


eval vieneu:   0%|          | 0/63 [00:00<?, ?it/s]

   WER=0.0392  CER=0.0197  (legacy WER=0.0487)  n=500

=== bud500 · data/corpora/bud500 [test] · n=500 ===


eval bud500:   0%|          | 0/63 [00:00<?, ?it/s]

   WER=0.0498  CER=0.0362  (legacy WER=0.0498)  n=500


## 4 · Save results + summary

In [6]:
pathlib.Path("results").mkdir(exist_ok=True)
for name, f in base_frames.items():
    f.to_csv(f"results/3ds_baseline_{name}_predictions.csv", index=False)
with open("results/3ds_baseline_metrics.json", "w", encoding="utf-8") as fh:
    json.dump(base_metrics, fh, ensure_ascii=False, indent=2)
print("saved -> results/3ds_baseline_metrics.json + results/3ds_baseline_<name>_predictions.csv")

summary = pd.DataFrame([{
    "corpus": name,
    "n": m["n"],
    "WER %": round(100 * m["wer"], 2),
    "CER %": round(100 * m["cer"], 2),
    "WER % (legacy)": round(100 * m["wer_legacy"], 2),
} for name, m in base_metrics.items()])
print()
print(summary.to_string(index=False))
summary

saved -> results/3ds_baseline_metrics.json + results/3ds_baseline_<name>_predictions.csv

      corpus   n  WER %  CER %  WER % (legacy)
vivoice_full 500   5.43   2.70            7.13
  vietspeech 500   6.59   3.76            6.62
      vieneu 500   3.92   1.97            4.87
      bud500 500   4.98   3.62            4.98


,corpus,n,WER %,CER %,WER % (legacy)
0,vivoice_full,500,5.43,2.70,7.13
1,vietspeech,500,6.59,3.76,6.62
2,vieneu,500,3.92,1.97,4.87
3,bud500,500,4.98,3.62,4.98


## 5 · Per-bucket breakdown

Duration matters: viVoice segments are 5–60 s while VietSpeech clips are mostly
2–6 s, so a single WER per corpus hides where the model actually struggles.
Short clips give the language-model prior less context to resolve ambiguity,
which usually shows up as a higher WER in the `0-5` bucket.

In [7]:
from vi_norm import wer_cer

rows = []
for name, f in base_frames.items():
    for bucket, g in f.groupby("bucket"):
        m = wer_cer(g["ref"], g["hyp"])
        rows.append({"corpus": name, "bucket": bucket, "n": m["n"],
                     "WER %": round(100 * m["wer"], 2),
                     "CER %": round(100 * m["cer"], 2)})
buckets = pd.DataFrame(rows).sort_values(["corpus", "bucket"])
print(buckets.to_string(index=False))
buckets

      corpus bucket   n  WER %  CER %
      bud500    0-5 500   4.98   3.62
      vieneu    0-5 169   3.98   2.28
      vieneu   5-30 331   3.90   1.89
  vietspeech    0-5 370   6.25   3.66
  vietspeech   5-30 130   7.13   3.92
vivoice_full    0-5 289   7.02   3.69
vivoice_full   5-30 211   4.55   2.17


,corpus,bucket,n,WER %,CER %
6,bud500,0-5,500,4.98,3.62
4,vieneu,0-5,169,3.98,2.28
5,vieneu,5-30,331,3.90,1.89
2,vietspeech,0-5,370,6.25,3.66
3,vietspeech,5-30,130,7.13,3.92
0,vivoice_full,0-5,289,7.02,3.69
1,vivoice_full,5-30,211,4.55,2.17


## 6 · Sanity check against notebook 1

Only fires when `NAMES` includes the cached `"vivoice"` entry, which scores the
same 114 segments as `01_eval_baseline.ipynb` — the two numbers should then
agree to the digit, and a mismatch means something in the shared path changed
(`vi_norm`, the batching, or the cache itself).

With the default `"vivoice_full"` there is nothing to compare: that entry
streams its own clips and builds its own splits, so it shares no rows with
notebook 1. The cell prints a note and moves on.

In [8]:
try:
    nb1 = json.load(open("results/baseline_metrics.json"))["overall"]
    here = base_metrics.get("vivoice")
    if here is None:
        print("viVoice was not evaluated in this run — nothing to compare.")
    else:
        delta = here["wer"] - nb1["wer"]
        print(f"notebook 1 viVoice WER: {nb1['wer']:.6f}  (n={nb1['n']})")
        print(f"notebook 3 viVoice WER: {here['wer']:.6f}  (n={here['n']})")
        print(f"delta: {delta:+.6f}")
        if here["n"] != nb1["n"]:
            print("NOTE: different n — EVAL_LIMIT is subsampling, so exact "
                  "agreement is not expected.")
        elif abs(delta) < 1e-9:
            print("OK — identical, as expected.")
        else:
            print("WARNING: same n but different WER. Investigate before "
                  "trusting notebook 4's before/after.")
except FileNotFoundError:
    print("No results/baseline_metrics.json — run 01_eval_baseline.ipynb "
          "to enable this check.")

viVoice was not evaluated in this run — nothing to compare.
